In [2]:
from resolver import Resolver

resolver = Resolver()

In [15]:
sample = resolver.rescued_data.sample(100)
sample

,dataset,dataset_id,status,url,source_website,organization,agency,download_date,size,maintainer,download_location,file_type,notes,metadata_available,metadata_url
2389,NCHS - Births and General Fertility Rates: United States,4598,Finished,https://data.cdc.gov/National-Center-for-Health-Statistics/NCHS-Births-and-General-Fertility-Rates-United-Sta/e6fc-ccez/about_data,data.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,2026-01-10,0.0,"DRP, DL",https://www.datalumos.org/datalumos/project/243401/version/V1/view,"PDF, CSV",<NA>,<NA>,<NA>
3695,"Soil Water Content Data for The Bushland, Texas Alfalfa Experiments",5771,Finished,https://agdatacommons.nal.usda.gov/articles/dataset/Soil_Water_Content_Data_for_The_Bushland_Texas_Alfalfa_Experiments/24856047,agdatacommons.nal.usda.gov,National Agricultural Library,U.S. Department of Agriculture,2026-06-28,0.0015,"DRP, DL",https://www.datalumos.org/datalumos/project/250770/version/V1/view,"HTML, JSON, XLSX",<NA>,yes,http://web.archive.org/web/20251017043844/https://agdatacommons.nal.usda.gov/articles/dataset/Soil_Water_Content_Data_for_The_Bushland_Texas_Alfalfa_Experiments/24856047
4052,Natality for 2007 - 2024 (expanded),6221,Finished,https://wonder.cdc.gov/natality-current.html,wonder.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,2026-07-01,0.1,"DRP, DL",https://www.datalumos.org/datalumos/project/251043/version/V1/view,CSV,<NA>,<NA>,<NA>
2056,"NNDSS - Table 1B. Arboviral diseases, Jamestown Canyon virus disease to Powassan virus disease (w46e-8kr3)",4066,Finished,https://data.cdc.gov/NNDSS/NNDSS-Table-1B-Arboviral-diseases-Jamestown-Canyon/w46e-8kr3/about_data,data.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,2026-01-16,0.0005,"DRP, DL",https://www.datalumos.org/datalumos/project/244341/version/V1/view,"PDF, CSV",<NA>,<NA>,<NA>
4226,"Trask River Watershed Study – Benthic macroinvertebrate sampling, 2006-2016",6403,Finished,https://www.fs.usda.gov/rds/archive/catalog/RDS-2022-0005,fs.usda.gov,US Forest Service,U.S. Department of Agriculture,2026-06-01,0.0127,"DRP, DL",https://www.datalumos.org/datalumos/project/249045/version/V1/view,"PDF, ZIP",<NA>,yes,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2197,NNDSS - Table II. Invasive Pneumococcal to Legionellosis (yqwx-bvu7),4207,Finished,https://data.cdc.gov/NNDSS/NNDSS-Table-II-Invasive-Pneumococcal-to-Legionello/yqwx-bvu7/about_data,data.cdc.gov,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,2026-01-12,0.0004,"DRP, DL",https://www.datalumos.org/datalumos/project/243989/version/V1/view,"PDF, CSV",<NA>,<NA>,<NA>
1299,NCVAS State Summary South Dakota FY2021,1451,Finished,https://www.data.va.gov/stories/s/v9rk-5gjz,data.va.gov,Office of Information and Technology - IT Operations and Services (ITOPS),Department of Veterans Affairs,2025-08-16,0.0015,"DRP, DL",https://doi.org/10.3886/E237245V1,"CSV, PDF",<NA>,<NA>,<NA>
3067,U.S. Bioenergy Statistics,5122,Finished,https://www.ers.usda.gov/data-products/us-bioenergy-statistics,ers.usda.gov,Economic Research Service,U.S. Department of Agriculture,2026-06-24,0.0012,"DRP, DL",https://www.datalumos.org/datalumos/project/250168/version/V1/view,ZIP,<NA>,<NA>,<NA>
4556,Fire Lab tree list—A tree-level model of the conterminous United States landscape circa 2014,6737,Finished,https://www.fs.usda.gov/rds/archive/catalog/RDS-2019-0026,fs.usda.gov,US Forest Service,U.S. Department of Agriculture,2026-06-05,4.5,"DRP, DL",https://www.datalumos.org/datalumos/project/249381/version/V1/view,"PDF, ZIP",<NA>,yes,<NA>


In [19]:
import asyncio
import re

import httpx
import pandas as pd


async def get_status(client: httpx.AsyncClient, url: str) -> int | str:
    # some URLs are missing the protocol
    if not re.search(r"^https?://", url):
        url = f"https://{url}"

    try:
        response = await client.head(url, follow_redirects=True)
        return response.status_code
    except httpx.TimeoutException:
        return "timeout"
    except httpx.HTTPError as e:
        return f"error: {e}"


async def get_statuses() -> list[int | str]:
    timeout = httpx.Timeout(20.0, connect=20.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        return await asyncio.gather(*(get_status(client, url) for url in sample["url"]))


statuses = await get_statuses()
sample["status"] = statuses


pd.set_option("display.max_colwidth", 200)
sample[["url", "status"]]


,url,status
2389,https://data.cdc.gov/National-Center-for-Health-Statistics/NCHS-Births-and-General-Fertility-Rates-United-Sta/e6fc-ccez/about_data,200
3695,https://agdatacommons.nal.usda.gov/articles/dataset/Soil_Water_Content_Data_for_The_Bushland_Texas_Alfalfa_Experiments/24856047,202
4052,https://wonder.cdc.gov/natality-current.html,200
2056,https://data.cdc.gov/NNDSS/NNDSS-Table-1B-Arboviral-diseases-Jamestown-Canyon/w46e-8kr3/about_data,200
4226,https://www.fs.usda.gov/rds/archive/catalog/RDS-2022-0005,timeout
...,...,...
2197,https://data.cdc.gov/NNDSS/NNDSS-Table-II-Invasive-Pneumococcal-to-Legionello/yqwx-bvu7/about_data,200
1299,https://www.data.va.gov/stories/s/v9rk-5gjz,200
3067,https://www.ers.usda.gov/data-products/us-bioenergy-statistics,200
4556,https://www.fs.usda.gov/rds/archive/catalog/RDS-2019-0026,timeout


In [20]:
sample[sample["status"] != 200][["url", "status"]]

,url,status
3695,https://agdatacommons.nal.usda.gov/articles/dataset/Soil_Water_Content_Data_for_The_Bushland_Texas_Alfalfa_Experiments/24856047,202
4226,https://www.fs.usda.gov/rds/archive/catalog/RDS-2022-0005,timeout
3258,https://agdatacommons.nal.usda.gov/articles/dataset/USDA_Agricultural_Research_Service_-_Patented_Bioenergy_and_Environment_Technologies/24661653,202
2887,https://data.cdc.gov/Environmental-Health-Toxicology/Daily-Census-Tract-Level-Ozone-Concentrations-2011/372p-dx3h/about_data,404
3660,https://agdatacommons.nal.usda.gov/articles/dataset/Drosophila_takahashii_genome_assembly_Dtak02082011/24855444,202
3136,https://www.fs.usda.gov/rds/archive/catalog/RDS-2025-0022,timeout
3579,https://agdatacommons.nal.usda.gov/articles/dataset/Feedstock_Readiness_Level_FSRL_evaluation_Pennisetum_purpureum_x_glaucum_banagrass_Alcohol-to-Jet_Hawaii_June_2018/24852846,202
4145,https://www.fs.usda.gov/rds/archive/catalog/RDS-2016-0025-2,timeout
3790,https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_The_survival_and_growth_of_honey_bee_Hymenoptera_Apidae_colonies_overwintered_in_cold_storage_the_effects_of_time_and_colony_location/...,202
1642,https://hifld-geoplatform.hub.arcgis.com/,404
